# Bulk RNA-seq Pipeline — Colab

Run the full Snakemake pipeline (DESeq2 · GSEA · ORA · TFEA · PROGENy · cMap → HTML report).

**What you need:**
nf-core/rnaseq outputs:
- `salmon.merged.gene_counts_length_scaled.tsv`
- `multiqc_report_data/` directory **zipped** (Colab uploads accept files, not folders)

**What you'll get:** `report/report.html` and `results.zip`

**Time budget:** ~10 min on first run, 1–2 min when rerunning in the same session.

## 1. Install conda + Snakemake

One-time setup for this session.

In [ ]:
%%time
# Skip download/install if a previous run already populated /content/miniforge.
import os
if not os.path.exists('/content/miniforge/bin/conda'):
    !wget -q https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh -O /tmp/miniforge.sh
    !bash /tmp/miniforge.sh -b -p /content/miniforge
    !rm /tmp/miniforge.sh

# Add miniforge to PATH so subprocesses spawned by Snakemake can find `conda`.
os.environ['PATH'] = '/content/miniforge/bin:' + os.environ['PATH']

!mamba install -y -n base -c conda-forge -c bioconda snakemake pandas pyyaml 2>&1 | tail -5
!snakemake --version

# ---------------------------------------------------------------------------
# HACK: swap Colab system libs for miniforge's newer ones.
#
# WHY: Colab's base image (as of 2026-04) ships Ubuntu libs that lag behind
# what conda-forge builds expect. Two known mismatches the rpy2 path in
# section 11 hits:
#   - libstdc++.so.6 lacks CXXABI_1.3.15 (needed by libicui18n.so.78)
#   - libssl/libcrypto.so.3 lack OPENSSL_3.2.0 (needed by libcurl.so.4)
# The kernel has already loaded these system libs by the time any cell runs,
# so an in-process ctypes preload can't override the SONAME — overwrite the
# files on disk and force-restart the kernel. Doing it here means the restart
# happens BEFORE the long pipeline run, not in the middle of section 11.
# Per-lib `.bak` sidecars make this idempotent across re-runs.
#
# WHEN TO REMOVE: when Colab's base image catches up. Verify on a fresh
# runtime, BEFORE this cell, that both check out:
#   !strings /usr/lib/x86_64-linux-gnu/libstdc++.so.6 | grep -c CXXABI_1.3.15
#   !strings /usr/lib/x86_64-linux-gnu/libssl.so.3    | grep -c OPENSSL_3.2.0
# Both >0 → delete this whole block (and the assert in section 11). The libs
# are ABI-backward-compatible so older Colab packages keep working.
# ---------------------------------------------------------------------------
import glob, shutil
def _swap_lib(libname):
    cands = sorted(c for c in glob.glob(f'/content/miniforge/lib/{libname}.*')
                   if os.path.isfile(c) and not os.path.islink(c))
    if not cands:
        return False
    dst = f'/usr/lib/x86_64-linux-gnu/{libname}'
    if os.path.exists(dst + '.bak'):
        return False
    if os.path.lexists(dst):
        shutil.move(dst, dst + '.bak')
    shutil.copy2(cands[-1], dst)
    print(f'swapped {libname} <- {cands[-1]}')
    return True

_did_swap = any([_swap_lib('libstdc++.so.6'),
                 _swap_lib('libssl.so.3'),
                 _swap_lib('libcrypto.so.3')])
if _did_swap:
    print('Restarting kernel — re-run this cell after the runtime comes back.')
    os.kill(os.getpid(), 9)


## 2. Get the pipeline code

In [ ]:
!rm -rf /content/bulk-rnaseq
!git clone --depth=1 https://github.com/artblakey19/BulkRNAseq-Analyzer.git /content/bulk-rnaseq
%cd /content/bulk-rnaseq


## 3. Upload your data

1. **Counts TSV** — `salmon.merged.gene_counts_length_scaled.tsv`
2. **MultiQC data** — `multiqc_report_data/` zipped into a single `.zip` (Colab cannot accept folders directly)

In [ ]:
# Upload counts TSV
from google.colab import files
uploaded = files.upload()
counts_file = next(iter(uploaded))
!mv "{counts_file}" /content/bulk-rnaseq/counts.tsv
!ls -la counts.tsv
print('Header:')
!head -1 counts.tsv | tr '\t' '\n'

In [ ]:
# Upload multiqc_report_data.zip and extract
from google.colab import files
uploaded = files.upload()
zip_file = next(iter(uploaded))
!rm -rf multiqc_report_data
!unzip -q "{zip_file}" -d /tmp/mqc_unpack
# Handle both "multiqc_report_data.zip containing multiqc_report_data/" and "zip of file contents directly"
!if [ -d /tmp/mqc_unpack/multiqc_report_data ]; then mv /tmp/mqc_unpack/multiqc_report_data /content/bulk-rnaseq/; else mv /tmp/mqc_unpack /content/bulk-rnaseq/multiqc_report_data; fi
!rm -f "{zip_file}"
!ls multiqc_report_data | head -10

## 4. Project metadata

Fill in metadata; appears in the HTML report header.

In [ ]:
#@markdown ### Project
project_name = "My RNA-seq study"  #@param {type:"string"}
analyst = ""  #@param {type:"string"}

#@markdown ### Experiment context (all optional)
cell_line = ""  #@param {type:"string"}
compound = ""  #@param {type:"string"}
dose = ""  #@param {type:"string"}
duration = ""  #@param {type:"string"}
notes = ""  #@param {type:"string"}

print('project metadata captured.')

## 5. Samples and contrasts

Two things to specify:

- **SAMPLES** — which **group (condition)** each sample belongs to. For 4 samples with the first two as controls and the last two treated: `{"Ctrl_1": "Control", "Ctrl_2": "Control", "Tx_1": "Treatment", "Tx_2": "Treatment"}`. Group names are free-form (DMSO/Drug, WT/KO, etc.).
- **CONTRASTS** — which groups to compare. Specify as `(target, reference)` pairs. The reference (control) must come second so that a positive fold-change means "upregulated in target vs. reference".

### Steps

1. **Run the first code cell below** — reads the header of your counts file and prints a SAMPLES template.
2. **Edit the second code cell** — paste the template and replace each `"CHANGE_ME"` with the sample's group name, then set `CONTRASTS` to the comparison(s) you want.

In [ ]:
# Detect samples from counts header (columns 3+ are samples)
from pathlib import Path

counts_path = Path('/content/bulk-rnaseq/counts.tsv')
if not counts_path.exists():
    # Fall back to test fixture for users who skipped upload
    counts_path = Path('/content/bulk-rnaseq/tests/test_data/counts.tsv')
    print('(using test fixture — no user data uploaded)')

header = counts_path.read_text().splitlines()[0].split('\t')
detected_samples = header[2:]
print(f'Detected {len(detected_samples)} samples:\n')
print('SAMPLES = {')
for s in detected_samples:
    print(f'    {s!r}: "CHANGE_ME",')
print('}')

In [ ]:
### ← EDIT ### — paste the template from the cell above, then set each condition.
SAMPLES = {
    "Control_1":   "Control",
    "Control_2":   "Control",
    "Treatment_1": "Treatment",
    "Treatment_2": "Treatment",
}

### ← EDIT ### — list every comparison you want (numerator vs denominator).
# numerator = the condition of interest (e.g. treated)
# denominator = the reference (e.g. control)
CONTRASTS = [
    # (contrast_id, numerator, denominator, description)
    ("Treatment_vs_Control", "Treatment", "Control", "Treatment vs Control"),
]

## 6. Generate config files

Writes `config/config.yaml`, `config/samples.tsv`, `config/contrasts.tsv` from the form inputs and SAMPLES/CONTRASTS above. Analysis settings (DE cutoffs, MSigDB collections, etc.) are taken from `config.template.yaml`.

In [ ]:
import csv
from collections import Counter
from pathlib import Path
import yaml

repo = Path('/content/bulk-rnaseq')
config_path = repo / 'config' / 'config.yaml'
samples_path = repo / 'config' / 'samples.tsv'
contrasts_path = repo / 'config' / 'contrasts.tsv'
template_path = repo / 'config' / 'config.template.yaml'

# Decide which counts/multiqc to use
user_counts = repo / 'counts.tsv'
user_mqc = repo / 'multiqc_report_data'
if user_counts.exists() and user_mqc.exists():
    counts_rel = 'counts.tsv'
    mqc_rel = 'multiqc_report_data'
    print('Using uploaded data.')
else:
    counts_rel = 'tests/test_data/counts.tsv'
    mqc_rel = 'tests/test_data/multiqc_report_data'
    print('Using bundled test fixture.')

cfg = yaml.safe_load(template_path.read_text())
cfg['project']['name'] = project_name
cfg['project']['analyst'] = analyst
cfg['project']['experiment'] = {
    'cell_line': cell_line, 'compound': compound, 'dose': dose,
    'duration': duration, 'notes': notes,
}
cfg['input']['counts_tsv'] = counts_rel
cfg['input']['multiqc_data_dir'] = mqc_rel
cfg['input']['samples_tsv'] = 'config/samples.tsv'
cfg['input']['contrasts_tsv'] = 'config/contrasts.tsv'

# Validate SAMPLES against counts header
header = (repo / counts_rel).read_text().splitlines()[0].split('\t')[2:]
missing = [s for s in header if s not in SAMPLES]
extra = [s for s in SAMPLES if s not in header]
if missing or extra:
    raise SystemExit(
        f'SAMPLES dict mismatch.\n'
        f'  samples in counts but not in SAMPLES: {missing}\n'
        f'  keys in SAMPLES but not in counts:   {extra}'
    )
unknown = [c for c in SAMPLES.values() if c == 'CHANGE_ME' or not c]
if unknown:
    raise SystemExit('fill in every SAMPLES value (no CHANGE_ME / empty allowed)')

# samples.tsv — auto-number replicates within each condition
replicate_of = Counter()
sample_rows = []
for sid in header:  # preserve column order
    cond = SAMPLES[sid]
    replicate_of[cond] += 1
    sample_rows.append({'sample': sid, 'condition': cond,
                         'replicate': replicate_of[cond], 'batch': 1})

# contrasts.tsv
conds = set(SAMPLES.values())
contrast_rows = []
for cid, num, den, desc in CONTRASTS:
    if num not in conds or den not in conds:
        raise SystemExit(f'contrast {cid}: {num!r} or {den!r} not in conditions {conds}')
    contrast_rows.append({'contrast_id': cid, 'factor': 'condition',
                           'numerator': num, 'denominator': den, 'description': desc})

config_path.parent.mkdir(parents=True, exist_ok=True)
with config_path.open('w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False, default_flow_style=False)

def write_tsv(p, rows, cols):
    with p.open('w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=cols, delimiter='\t')
        w.writeheader(); w.writerows(rows)

write_tsv(samples_path, sample_rows, ['sample', 'condition', 'replicate', 'batch'])
write_tsv(contrasts_path, contrast_rows,
          ['contrast_id', 'factor', 'numerator', 'denominator', 'description'])

print('wrote:')
print(f'  {config_path.relative_to(repo)}')
print(f'  {samples_path.relative_to(repo)} ({len(sample_rows)} rows)')
print(f'  {contrasts_path.relative_to(repo)} ({len(contrast_rows)} rows)')

## 7. Dry-run (verifies the DAG without running anything, optional)

In [ ]:
!snakemake \
  --snakefile workflow/Snakefile \
  --configfile config/config.yaml \
  --dry-run 2>&1 | tail -30

## 8. Run the pipeline

First run: ~10 min (conda env installation). Rerunning in the same Colab session: 1–2 min. The `--keep-going` flag continues past failed rules (e.g. L2S2 network hiccup) so the rest of the pipeline still runs.

In [ ]:
%%time
#@markdown ### Run options
cores = 2  #@param {type:"slider", min:1, max:8, step:1}
keep_going = True  #@param {type:"boolean"}

_flag = "--keep-going" if keep_going else ""
!snakemake --snakefile workflow/Snakefile --configfile config/config.yaml --use-conda --cores {cores} {_flag} 2>&1 | tail -80

## 9. Download results

Bundles everything under `results/` into a single zip. The HTML report is at `report/report.html` inside it.

In [ ]:
from pathlib import Path
from google.colab import files

report = Path('results/report/report.html')
if not report.exists():
    raise SystemExit('report.html missing — check section 8 output or section 10 logs')

!zip -qr results.zip results/
print(f'report size: {report.stat().st_size/1024:.1f} KB')
files.download('results.zip')

## 10. Troubleshooting

Dumps every per-rule log so you can see which rule failed and why.

In [ ]:
!for f in $(find logs -name '*.log' 2>/dev/null); do echo "=== $f ==="; tail -30 "$f"; echo; done

## 11. Explore — replot with custom cutoffs

After the pipeline finishes, `results/` contains everything needed to iterate on plots. This section hosts R inside the Python kernel via `rpy2`'s `%%R` magic, reusing the r-deseq2 conda env the pipeline already built under `.snakemake/conda/`. No extra conda installs (except a small add-on for the GSEA enrichment plot + L2S2 re-query further down).

Tweak `padj_cutoff` / `lfc_cutoff`, top-N counts, GSEA pathway names, L2S2 signature size, etc. — all without re-running the pipeline.

### Setup — install rpy2 + load R env

In [ ]:
import glob, os, subprocess
# Find a conda env built by the pipeline that has SummarizedExperiment (r-deseq2).
def has_pkg(r_bin, pkg):
    try:
        subprocess.check_call(
            [r_bin, '-e', f'suppressMessages(library({pkg}))'],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        return True
    except Exception:
        return False

r_bins = sorted(glob.glob('/content/bulk-rnaseq/.snakemake/conda/*/bin/R'))
R_BIN = next((r for r in r_bins if has_pkg(r, 'SummarizedExperiment')), None)
assert R_BIN, 'No conda env with SummarizedExperiment found — run section 8 first.'

# Sanity check: section 1 should have already swapped the system libs and
# restarted the kernel. The .bak sidecar from that swap is our marker.
# If missing, rpy2 / curl will fail with CXXABI/OPENSSL version errors.
assert os.path.exists('/usr/lib/x86_64-linux-gnu/libstdc++.so.6.bak'), (
    'system lib swap missing — re-run section 1 (the kernel will restart once).')

# Restore miniforge on PATH (section 1's kernel restart wiped os.environ).
# Subsequent cells — including the GSEA fgsea-install cell — assume `mamba`
# resolves on PATH.
os.environ['PATH'] = '/content/miniforge/bin:' + os.environ['PATH']

# r-deseq2 env carries only DESeq2/SummarizedExperiment/apeglm/matrixStats —
# the pipeline's exploratory.R uses base R. The Explore plots below need the
# tidyverse stack, so install whatever's missing into the same env on demand.
ENV_PREFIX = os.path.dirname(os.path.dirname(R_BIN))
need = [p for p in ['yaml', 'readr', 'dplyr', 'tidyr', 'ggplot2', 'scales', 'forcats']
        if not has_pkg(R_BIN, p)]
if need:
    print(f'Installing into {ENV_PREFIX}: {need}')
    subprocess.check_call(
        ['mamba', 'install', '-y', '--prefix', ENV_PREFIX,
         '-c', 'conda-forge'] + [f'r-{p}' for p in need])

R_HOME = os.path.realpath(os.path.join(os.path.dirname(R_BIN), '..', 'lib', 'R'))
os.environ['R_HOME'] = R_HOME
os.environ['PATH']   = f'{os.path.dirname(R_BIN)}:' + os.environ['PATH']
print('R_HOME =', R_HOME)

!pip install --quiet rpy2
%load_ext rpy2.ipython


### R-side setup (runs once)

In [ ]:
%%R
suppressPackageStartupMessages({
  library(yaml); library(readr); library(dplyr); library(tidyr)
  library(ggplot2); library(scales)
})
`%||%` <- function(a, b) if (is.null(a)) b else a

PROJECT_ROOT <- "/content/bulk-rnaseq"
rp <- function(...) file.path(PROJECT_ROOT, ...)
cfg       <- yaml::read_yaml(rp("config", "config.yaml"))
samples   <- read_tsv(rp(cfg$input$samples_tsv   %||% "config/samples.tsv"),   show_col_types = FALSE)
contrasts <- read_tsv(rp(cfg$input$contrasts_tsv %||% "config/contrasts.tsv"), show_col_types = FALSE)
CONTRAST <- contrasts$contrast_id[1]

safe_read <- function(path, sep = ",") {
  if (is.null(path) || !file.exists(path)) return(NULL)
  df <- tryCatch(
    if (sep == "\t") read_tsv(path, show_col_types = FALSE)
    else             read_csv(path, show_col_types = FALSE),
    error = function(e) NULL)
  if (is.null(df) || nrow(df) == 0) return(df)
  df
}
no_results <- function(msg) { message(msg); invisible(NULL) }
.trunc <- function(x, n = 50) ifelse(nchar(x) > n, paste0(substr(x, 1, n-1), "..."), x)

cat("Contrast:", CONTRAST, "\nAvailable:", paste(contrasts$contrast_id, collapse = ", "), "\n")


### QC

In [ ]:
%%R -w 700 -h 380
qc <- safe_read(rp("results", "qc", "qc_summary.tsv"), sep = "\t")
if (is.null(qc) || !"assigned_reads" %in% names(qc)) {
  no_results("QC summary not available.")
} else {
  df <- qc |> dplyr::mutate(sample = factor(sample, levels = sample),
                            condition = as.character(condition))
  ggplot(df, aes(sample, assigned_reads, fill = condition)) +
    geom_col() + scale_y_continuous(labels = scales::comma) +
    labs(x = NULL, y = "Assigned reads") +
    theme_minimal(base_size = 11) +
    theme(axis.text.x = element_text(angle = 45, hjust = 1))
}


### Exploratory — PCA / scree / sample distance

In [ ]:
%%R -w 750 -h 550
pca_obj <- readRDS(rp("results", "exploratory", "pca.rds"))
scores <- pca_obj$scores
ve <- pca_obj$var_explained
ggplot(scores, aes(PC1, PC2, colour = condition, shape = as.factor(batch), label = sample)) +
  geom_point(size = 3) +
  geom_text(nudge_y = 0.04, size = 3, show.legend = FALSE) +
  labs(x = sprintf("PC1 (%.1f%%)", 100 * (ve[["PC1"]] %||% NA)),
       y = sprintf("PC2 (%.1f%%)", 100 * (ve[["PC2"]] %||% NA)),
       shape = "batch") +
  theme_minimal(base_size = 12)


In [ ]:
%%R -w 750 -h 400
ve <- pca_obj$var_explained
scree <- head(data.frame(PC = factor(names(ve), levels = names(ve)),
                          pct = as.numeric(ve) * 100),
              min(10, length(ve)))
ggplot(scree, aes(PC, pct)) + geom_col(fill = "#4c7fb0") +
  labs(x = NULL, y = "% variance explained") +
  theme_minimal(base_size = 12)


In [ ]:
%%R -w 750 -h 650
cor_obj <- readRDS(rp("results", "exploratory", "sample_correlation.rds"))
M <- cor_obj$dist
ord <- if (!is.null(cor_obj$hclust)) cor_obj$hclust$order else seq_len(nrow(M))
M <- M[ord, ord, drop = FALSE]
df <- as.data.frame(as.table(M))
names(df) <- c("x", "y", "distance")
df$x <- factor(df$x, levels = rownames(M)); df$y <- factor(df$y, levels = rownames(M))
ggplot(df, aes(x, y, fill = distance)) +
  geom_tile() +
  geom_text(aes(label = sprintf("%.1f", distance)), size = 3, colour = "grey20") +
  scale_fill_gradient(low = "#4c7fb0", high = "#ffffff",
                      limits = c(0, max(df$distance, na.rm = TRUE))) +
  labs(x = NULL, y = NULL, fill = "distance") +
  theme_minimal(base_size = 12) +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))


### DE — volcano · MA · top-30 heatmap

Edit `padj_cutoff` / `lfc_cutoff` below to re-threshold without re-running DESeq2.

In [ ]:
#@markdown ### DE cutoff
padj_cutoff = 0.05  #@param {type:"number"}
lfc_cutoff = 1.0  #@param {type:"number"}

In [ ]:
%%R -i padj_cutoff -i lfc_cutoff
de_tbl <- safe_read(rp("results", "de", CONTRAST, "deseq2_results.csv"))
dds <- readRDS(rp("results", "exploratory", "dds_vst.rds"))
vst_mat <- tryCatch(SummarizedExperiment::assay(dds),
                    error = function(e) as.matrix(dds@assays@data[[1]]))

summary_path <- rp("results", "de", CONTRAST, "deg_summary.tsv")
safe_read(summary_path, sep = "\t")


In [ ]:
%%R -w 850 -h 600
v_df <- de_tbl |>
  dplyr::mutate(neglog10p = -log10(pmax(pvalue, 1e-300)),
                sig = !is.na(padj) & padj < padj_cutoff & abs(log2FoldChange) >= lfc_cutoff)
sig_df <- v_df |> dplyr::filter(sig)
ns_df  <- v_df |> dplyr::filter(!sig)
if (nrow(ns_df) > 3000) { set.seed(42); ns_df <- ns_df[sample.int(nrow(ns_df), 3000), ] }

ggplot() +
  geom_point(data = ns_df,  aes(log2FoldChange, neglog10p),
             colour = "grey70", alpha = 0.45, size = 1.1) +
  geom_point(data = sig_df, aes(log2FoldChange, neglog10p),
             colour = "#c0392b", alpha = 0.85, size = 1.6) +
  geom_vline(xintercept = c(-lfc_cutoff, lfc_cutoff), linetype = "dashed", colour = "grey40") +
  geom_hline(yintercept = -log10(padj_cutoff),       linetype = "dashed", colour = "grey40") +
  labs(title = CONTRAST, x = "log2FoldChange", y = "-log10(pvalue)",
       subtitle = sprintf("sig: padj<%.2g & |LFC|>=%.2g", padj_cutoff, lfc_cutoff)) +
  theme_minimal(base_size = 12)


In [ ]:
%%R -w 850 -h 900
top <- de_tbl |>
  dplyr::filter(!is.na(padj), !is.na(log2FoldChange)) |>
  dplyr::arrange(padj) |>
  head(30)

rows <- intersect(top$gene_id, rownames(vst_mat))
M <- vst_mat[rows, , drop = FALSE]
lbl <- setNames(top$gene_name, top$gene_id)
rn  <- ifelse(is.na(lbl[rows]) | !nzchar(lbl[rows]), rows, lbl[rows])
rownames(M) <- make.unique(rn)
Z <- t(scale(t(M)))
row_hc <- hclust(dist(Z),    method = "average")
col_hc <- hclust(dist(t(Z)), method = "average")
long <- as.data.frame(as.table(Z)); names(long) <- c("gene", "sample", "z")
long$gene   <- factor(long$gene,   levels = rownames(Z)[row_hc$order])
long$sample <- factor(long$sample, levels = colnames(Z)[col_hc$order])

ggplot(long, aes(sample, gene, fill = z)) +
  geom_tile() +
  scale_fill_gradient2(low = "#2c7bb6", mid = "white", high = "#d7191c", midpoint = 0) +
  labs(x = NULL, y = NULL, fill = "z-score") +
  theme_minimal(base_size = 11) +
  theme(axis.text.x = element_text(angle = 45, hjust = 1),
        axis.text.y = element_text(size = 9))


### GSEA — top 10 up / down per collection

In [ ]:
%%R -w 900 -h 600
gsea <- safe_read(rp("results", "enrichment", CONTRAST, "gsea_combined.csv"))
if (is.null(gsea) || nrow(gsea) == 0) {
  no_results("GSEA returned no results.")
} else {
  .coll <- cfg$enrichment$gsea$collections %||% list()
  .lbl  <- setNames(vapply(.coll, function(x) x$label %||% x$id, character(1)),
                    vapply(.coll, function(x) x$id, character(1)))
  coll_lbl <- function(id) { out <- unname(.lbl[id]); ifelse(is.na(out), id, out) }
  for (c in unique(as.character(gsea$collection))) {
    df <- gsea |> dplyr::filter(collection == c) |>
      dplyr::mutate(direction = ifelse(NES > 0, "up", "down")) |>
      dplyr::group_by(direction) |>
      dplyr::arrange(padj, .by_group = TRUE) |>
      dplyr::slice_head(n = 10) |>
      dplyr::ungroup() |>
      dplyr::mutate(label = .trunc(pathway))
    if (nrow(df) == 0) next
    df$label <- factor(df$label, levels = rev(unique(df$label)))
    print(ggplot(df, aes(NES, label, fill = direction)) +
      geom_col() +
      scale_fill_manual(values = c(up = "#c0392b", down = "#2c7bb6")) +
      geom_vline(xintercept = 0, colour = "grey40") +
      labs(x = "NES", y = NULL,
           title = sprintf("%s — Top 10 up / Top 10 down", coll_lbl(c))) +
      theme_minimal(base_size = 11))
  }
}


### ORA — top 10 up / down per database

In [ ]:
%%R -w 900 -h 700
ora <- safe_read(rp("results", "enrichment", CONTRAST, "ora_combined.csv"))
if (is.null(ora) || nrow(ora) == 0) {
  no_results("ORA returned no results.")
} else {
  .db <- cfg$enrichment$ora$databases %||% list()
  .lbl <- setNames(vapply(.db, function(x) x$label %||% x$id, character(1)),
                   vapply(.db, function(x) x$id, character(1)))
  db_lbl <- function(id) { out <- unname(.lbl[id]); ifelse(is.na(out), id, out) }
  for (db in unique(ora$database)) {
    df <- ora |> dplyr::filter(database == db, direction %in% c("up", "down")) |>
      dplyr::group_by(direction) |>
      dplyr::arrange(p.adjust, .by_group = TRUE) |>
      dplyr::slice_head(n = 10) |>
      dplyr::ungroup() |>
      dplyr::mutate(
        neglog10p   = -log10(pmax(p.adjust, 1e-300)),
        Description = ifelse(is.na(Description) | !nzchar(Description), ID, Description),
        Description = .trunc(Description, 50))
    if (nrow(df) == 0) next
    print(ggplot(df, aes(neglog10p, reorder(Description, neglog10p), fill = direction)) +
      geom_col(position = position_dodge(width = 0.8)) +
      scale_fill_manual(values = c(up = "#c0392b", down = "#2c7bb6")) +
      labs(x = "-log10(p.adjust)", y = NULL,
           title = sprintf("%s — Top 10 up / Top 10 down", db_lbl(db))) +
      theme_minimal(base_size = 11))
  }
}


### TFEA — top 30 TFs by |score|

In [ ]:
%%R -w 700 -h 800
tf <- safe_read(rp("results", "tfea", CONTRAST, "tf_scores.tsv"), sep = "\t")
if (is.null(tf) || nrow(tf) == 0) {
  no_results("TFEA results not available.")
} else {
  top <- tf |> dplyr::slice_max(abs(score), n = 30) |>
    dplyr::mutate(tf = forcats::fct_reorder(source, score))
  ggplot(top, aes(score, tf, fill = score > 0)) +
    geom_col() +
    scale_fill_manual(values = c(`TRUE` = "#c0392b", `FALSE` = "#2c7bb6"), guide = "none") +
    labs(title = paste("Top TFEA —", CONTRAST), y = NULL) +
    theme_minimal(base_size = 11)
}


### PROGENy — pathway activity (contrast-level, MLM on Wald stat)

Bar chart of MLM activity scores per pathway. Positive → up in numerator; negative → up in denominator.

In [ ]:
%%R -w 750 -h 600
pw <- safe_read(rp("results", "progeny", CONTRAST, "progeny_scores.tsv"), sep = "\t")
if (is.null(pw) || nrow(pw) == 0 ||
    !all(c("pathway", "score") %in% names(pw)) ||
    all(is.na(pw$score))) {
  no_results("PROGENy scores unavailable.")
} else {
  df <- pw |>
    dplyr::filter(!is.na(score)) |>
    dplyr::mutate(direction = ifelse(score > 0, "up", "down"),
                  pathway   = factor(pathway, levels = pathway[order(score)]))
  ggplot(df, aes(x = score, y = pathway, fill = direction)) +
    geom_col() +
    scale_fill_manual(values = c(up = "#c0392b", down = "#2c7bb6"), guide = "none") +
    geom_vline(xintercept = 0, colour = "grey40") +
    labs(x = "MLM activity score", y = NULL,
         title = "PROGENy pathway activity",
         subtitle = sprintf("%s — input: DESeq2 Wald stat", CONTRAST)) +
    theme_minimal(base_size = 12)
}


#### Score table (sorted by |score|)

In [ ]:
%%R
if (is.null(pw) || nrow(pw) == 0 || !"score" %in% names(pw)) {
  no_results("PROGENy table unavailable.")
} else {
  pw |> dplyr::arrange(dplyr::desc(abs(score)))
}

### cMap — top 30 perturbagens

#### L2S2 input gene signatures

In [ ]:
%%R
top_up   <- cfg$cmap$top_up   %||% 250
top_down <- cfg$cmap$top_down %||% 250

if (is.null(de_tbl) || nrow(de_tbl) == 0) {
  no_results("DE results unavailable; L2S2 input signature cannot be reconstructed.")
} else {
  df <- de_tbl |>
    dplyr::filter(!is.na(gene_name), !is.na(log2FoldChange), !is.na(stat),
                  nzchar(as.character(gene_name)))
  up_tbl <- df |> dplyr::filter(log2FoldChange > 0) |>
    dplyr::slice_max(order_by = stat, n = top_up, with_ties = FALSE) |>
    dplyr::distinct(gene_name, .keep_all = TRUE) |>
    dplyr::mutate(direction = "up", input_rank = dplyr::row_number())
  down_tbl <- df |> dplyr::filter(log2FoldChange < 0) |>
    dplyr::slice_min(order_by = stat, n = top_down, with_ties = FALSE) |>
    dplyr::distinct(gene_name, .keep_all = TRUE) |>
    dplyr::mutate(direction = "down", input_rank = dplyr::row_number())
  sig_tbl <- dplyr::bind_rows(up_tbl, down_tbl) |>
    dplyr::select(dplyr::any_of(c("direction", "input_rank", "gene_id", "gene_name",
                                   "log2FoldChange", "stat", "pvalue", "padj")))
  cat(sprintf("L2S2 input: %d up / %d down genes (ranked by |Wald stat|)\n",
              sum(sig_tbl$direction == "up"), sum(sig_tbl$direction == "down")))
  sig_tbl
}


In [ ]:
%%R -w 850 -h 800
hits <- safe_read(rp("results", "cmap", CONTRAST, "l2s2_hits.tsv"), sep = "\t")
if (is.null(hits) || nrow(hits) == 0) {
  no_results("L2S2 query returned no hits.")
} else {
  top <- hits |>
    dplyr::arrange(dplyr::desc(abs(score %||% 0)), fdr) |>
    head(30) |>
    dplyr::mutate(label = ifelse(is.na(perturbagen) | !nzchar(perturbagen),
                                 paste0("row_", dplyr::row_number()), perturbagen),
                  label = factor(label, levels = rev(label)))
  ggplot(top, aes(score, label, fill = ifelse(score > 1, "induce", "reverse"))) +
    geom_col() +
    scale_fill_manual(values = c(induce = "#e67e22", reverse = "#2c7bb6"), name = "direction") +
    geom_vline(xintercept = 1, linetype = "dashed", colour = "grey40") +
    labs(x = "odds ratio (score)", y = NULL, title = "Top 30 perturbagens") +
    theme_minimal(base_size = 11)
}


### GSEA enrichment plot (interactive)

Replots a single MSigDB pathway's enrichment curve. Requires `fgsea` and `msigdbr`, which live in the `r-enrichment` conda env — the cell below installs them into the active R env if missing (~1–2 min, one-time).

In [ ]:
import subprocess, os
env_prefix = os.path.dirname(os.path.dirname(R_BIN))   # .../envs/<hash>
need = []
for pkg in ['fgsea', 'msigdbr', 'httr', 'jsonlite']:
    try:
        subprocess.check_call([R_BIN, '-e', f'suppressMessages(library({pkg}))'],
                              stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    except subprocess.CalledProcessError:
        need.append(pkg)

if need:
    conda_pkg = {'fgsea': 'bioconductor-fgsea', 'msigdbr': 'r-msigdbr',
                 'httr':  'r-httr',             'jsonlite': 'r-jsonlite'}
    print(f'Installing into {env_prefix}: {need}')
    subprocess.check_call(
        ['mamba', 'install', '-y', '--prefix', env_prefix,
         '-c', 'conda-forge', '-c', 'bioconda'] + [conda_pkg[p] for p in need])
else:
    print('fgsea / msigdbr / httr / jsonlite already available.')


In [ ]:
%%R
suppressPackageStartupMessages({ library(fgsea); library(msigdbr) })

.ranking  <- cfg$enrichment$gsea$ranking %||% "stat"
.species  <- cfg$enrichment$species      %||% "Homo sapiens"
.gsea_combined <- safe_read(rp("results", "enrichment", CONTRAST, "gsea_combined.csv"))

.build_ranks <- function(de_res, method) {
  if (method == "stat") {
    rdf <- de_res |> dplyr::filter(!is.na(gene_name), !is.na(stat)) |>
      dplyr::group_by(gene_name) |> dplyr::summarise(metric = mean(stat), .groups = "drop")
  } else if (method == "signed_p") {
    rdf <- de_res |> dplyr::filter(!is.na(gene_name), !is.na(log2FoldChange), !is.na(pvalue)) |>
      dplyr::mutate(metric = sign(log2FoldChange) * -log10(pmax(pvalue, 1e-300))) |>
      dplyr::filter(is.finite(metric)) |>
      dplyr::group_by(gene_name) |> dplyr::summarise(metric = mean(metric), .groups = "drop")
  } else stop("Unsupported ranking: ", method)
  setNames(dplyr::arrange(rdf, dplyr::desc(metric))$metric,
           dplyr::arrange(rdf, dplyr::desc(metric))$gene_name)
}

.get_pathways <- function(coll_str, species) {
  parts   <- unlist(strsplit(coll_str, ":"))
  coll    <- parts[1]
  subcoll <- if (length(parts) > 1) paste(parts[-1], collapse = ":") else NULL
  m <- if (!is.null(subcoll)) msigdbr(species = species, collection = coll, subcollection = subcoll)
       else                   msigdbr(species = species, collection = coll)
  split(m$gene_symbol, m$gs_name)
}

.pathway_cache <- new.env(parent = emptyenv())

.detect_collection <- function(pathway) {
  prefixes <- c(HALLMARK_ = "H", REACTOME_ = "C2:CP:REACTOME",
                WP_ = "C2:CP:WIKIPATHWAYS", PID_ = "C2:CP:PID",
                BIOCARTA_ = "C2:CP:BIOCARTA")
  for (p in names(prefixes)) if (startsWith(pathway, p)) return(unname(prefixes[[p]]))
  if (!is.null(.gsea_combined)) {
    hit <- .gsea_combined[.gsea_combined$pathway == pathway, "collection", drop = TRUE]
    if (length(hit) > 0) return(as.character(hit[1]))
  }
  stop(sprintf("Could not infer collection for '%s'.", pathway))
}

.ranks <- .build_ranks(de_tbl, .ranking)

plot_gsea <- function(pathway, collection = NULL) {
  if (is.null(collection)) collection <- .detect_collection(pathway)
  if (is.null(.pathway_cache[[collection]]))
    .pathway_cache[[collection]] <- .get_pathways(collection, .species)
  sets <- .pathway_cache[[collection]]
  if (!pathway %in% names(sets))
    stop(sprintf("'%s' not found in collection %s.", pathway, collection))
  genes <- sets[[pathway]]
  row <- if (!is.null(.gsea_combined))
    .gsea_combined[.gsea_combined$collection == collection & .gsea_combined$pathway == pathway, ]
  else NULL
  nes_padj <- if (!is.null(row) && nrow(row) == 1)
    sprintf(" · NES=%.2f · padj=%.2g", row$NES, row$padj) else ""
  fgsea::plotEnrichment(genes, .ranks) +
    ggplot2::labs(title = pathway,
                  subtitle = sprintf("%s · %s · ranking=%s · |set ∩ ranked|=%d%s",
                                     CONTRAST, collection, .ranking,
                                     sum(genes %in% names(.ranks)), nes_padj))
}

list_pathways <- function(query = NULL, n = 20) {
  if (is.null(.gsea_combined) || nrow(.gsea_combined) == 0) return(.gsea_combined)
  all <- .gsea_combined |> dplyr::select(collection, pathway, NES, padj, size)
  if (!is.null(query)) all <- all[grepl(query, all$pathway, ignore.case = TRUE), ]
  dplyr::arrange(all, padj) |> head(n)
}

show_leading <- function(pathway, collection = NULL) {
  if (is.null(collection)) collection <- .detect_collection(pathway)
  if (is.null(.gsea_combined)) { message("GSEA combined CSV not found."); return(invisible()) }
  row <- .gsea_combined[.gsea_combined$collection == collection & .gsea_combined$pathway == pathway, ]
  if (nrow(row) == 0) { message("Pathway not in saved GSEA results."); return(invisible()) }
  le <- strsplit(as.character(row$leadingEdge[1]), ";", fixed = TRUE)[[1]]
  cat(sprintf("%s · %s\nNES=%.3f · padj=%.3g · size=%d · leadingEdge (%d):\n\n",
              pathway, collection, row$NES, row$padj, row$size, length(le)))
  cat(paste(le, collapse = ", "), "\n")
}
invisible(NULL)


Change the pathway name in `plot_gsea(...)` below to replot any pathway present in `gsea_combined.csv`. Collection auto-detected from the name prefix.

In [ ]:
#@markdown ### GSEA pathway
gsea_pathway = "HALLMARK_INTERFERON_GAMMA_RESPONSE"  #@param {type:"string"}
#@markdown Override only when collection auto-detection fails (e.g. `C2:CGP`)
gsea_collection_override = ""  #@param {type:"string"}


In [ ]:
%%R -w 800 -h 500 -i gsea_pathway -i gsea_collection_override
if (nzchar(gsea_collection_override)) {
  plot_gsea(gsea_pathway, collection = gsea_collection_override)
} else {
  plot_gsea(gsea_pathway)
}


Browse the pathways actually available for this contrast — filter by keyword, list leading-edge genes.

In [ ]:
%%R
list_pathways()                    # top 20 by padj across collections
# list_pathways("interferon")      # keyword filter
# list_pathways("apop", n = 50)
# show_leading("HALLMARK_INTERFERON_GAMMA_RESPONSE")


### L2S2 re-query (custom signature size)

Re-runs the L2S2 paired-enrichment query against the public API with a user-chosen signature size.

In [ ]:
%%R
## --- L2S2 re-query helpers (run once) ----------------------------------
suppressPackageStartupMessages({ library(httr); library(jsonlite) })

L2S2_ENDPOINT <- Sys.getenv("L2S2_API_URL", "https://l2s2.maayanlab.cloud/graphql")

.l2s2_paired_query <- '
query PairEnrichmentQuery($genesUp: [String]!, $genesDown: [String]!) {
  currentBackground {
    pairedEnrich(genesUp: $genesUp, genesDown: $genesDown) {
      consensus { drug oddsRatio pvalue adjPvalue }
    }
  }
}'

.l2s2_post <- function(payload) {
  resp <- httr::POST(L2S2_ENDPOINT, httr::content_type_json(),
                     body = jsonlite::toJSON(payload, auto_unbox = TRUE))
  httr::stop_for_status(resp)
  j <- httr::content(resp, as = "parsed", simplifyVector = FALSE)
  if (!is.null(j$errors))
    stop("GraphQL errors: ", jsonlite::toJSON(j$errors, auto_unbox = TRUE))
  j
}

# L2S2's pairedEnrich returns drug names only; MoA lives on FdaCount and the
# web UI joins them client-side. Mirror that with one aliased fdaCount batch.
.l2s2_fetch_moas <- function(drugs) {
  if (!length(drugs)) return(setNames(character(), character()))
  parts <- vapply(seq_along(drugs), function(i)
    sprintf('a%d: fdaCount(perturbation: %s) { moa }',
            i - 1L, jsonlite::toJSON(drugs[i], auto_unbox = TRUE)),
    character(1))
  query <- paste0("query MoaLookup { ", paste(parts, collapse = " "), " }")
  j <- tryCatch(.l2s2_post(list(query = query)),
                error = function(e) { message("MoA lookup failed: ", conditionMessage(e)); NULL })
  if (is.null(j)) return(setNames(rep("", length(drugs)), drugs))
  vals <- vapply(seq_along(drugs), function(i) {
    moa <- j$data[[paste0("a", i - 1L)]]$moa
    if (is.null(moa)) "" else as.character(moa)
  }, character(1))
  setNames(vals, drugs)
}

query_l2s2 <- function(top_up = 250, top_down = 250) {
  if (is.null(de_tbl)) stop("de_tbl not loaded — run the DE section first.")
  df <- de_tbl |>
    dplyr::filter(!is.na(gene_name), nzchar(as.character(gene_name)),
                  !is.na(log2FoldChange), !is.na(stat))
  up <- df |> dplyr::filter(log2FoldChange > 0) |>
    dplyr::slice_max(order_by = stat, n = top_up, with_ties = FALSE) |>
    dplyr::pull(gene_name) |> as.character() |> unique()
  down <- df |> dplyr::filter(log2FoldChange < 0) |>
    dplyr::slice_min(order_by = stat, n = top_down, with_ties = FALSE) |>
    dplyr::pull(gene_name) |> as.character() |> unique()
  if (!length(up) || !length(down))
    stop(sprintf("signature incomplete (up=%d down=%d); paired enrichment needs both.",
                 length(up), length(down)))
  cat(sprintf("Signature: %d up / %d down — querying %s\n",
              length(up), length(down), L2S2_ENDPOINT))

  payload <- list(query = .l2s2_paired_query,
                  variables = list(genesUp = I(up), genesDown = I(down)))
  j <- .l2s2_post(payload)
  cons <- j$data$currentBackground$pairedEnrich$consensus
  if (length(cons) == 0) { cat("No consensus hits.\n"); return(data.frame()) }

  out <- do.call(rbind, lapply(cons, function(r) data.frame(
    perturbagen = r$drug      %||% NA_character_,
    score       = r$oddsRatio %||% NA_real_,
    pvalue      = r$pvalue    %||% NA_real_,
    fdr         = r$adjPvalue %||% NA_real_,
    stringsAsFactors = FALSE)))
  out <- out[order(out$fdr, -out$score), , drop = FALSE]
  rownames(out) <- NULL

  drugs <- unique(out$perturbagen[!is.na(out$perturbagen) & nzchar(out$perturbagen)])
  moa_map <- .l2s2_fetch_moas(drugs)
  out$moa <- ifelse(out$perturbagen %in% names(moa_map),
                    moa_map[out$perturbagen], "")
  out$rank <- seq_len(nrow(out))
  out[, c("rank", "perturbagen", "moa", "score", "pvalue", "fdr")]
}

plot_l2s2 <- function(hits, n = 30, title = NULL) {
  if (is.null(hits) || nrow(hits) == 0) { no_results("No L2S2 hits."); return(invisible()) }
  top <- head(hits, n) |>
    dplyr::mutate(label = ifelse(is.na(perturbagen) | !nzchar(perturbagen),
                                 paste0("row_", dplyr::row_number()), perturbagen),
                  label = factor(label, levels = rev(label)))
  ggplot(top, aes(score, label, fill = ifelse(score > 1, "induce", "reverse"))) +
    geom_col() +
    scale_fill_manual(values = c(induce = "#e67e22", reverse = "#2c7bb6"), name = "direction") +
    geom_vline(xintercept = 1, linetype = "dashed", colour = "grey40") +
    labs(x = "odds ratio (score)", y = NULL,
         title = title %||% sprintf("Top %d perturbagens", nrow(top))) +
    theme_minimal(base_size = 11)
}

invisible(NULL)


In [ ]:
#@markdown ### L2S2 signature size
top_up = 100  #@param {type:"slider", min:50, max:500, step:50}
top_down = 100  #@param {type:"slider", min:50, max:500, step:50}
top_n_plot = 30  #@param {type:"slider", min:10, max:100, step:5}


In [ ]:
%%R -w 850 -h 800 -i top_up -i top_down -i top_n_plot
hits_custom <- query_l2s2(top_up = top_up, top_down = top_down)
plot_l2s2(hits_custom, n = top_n_plot,
          title = sprintf("Top %d perturbagens (up=%d, down=%d)", top_n_plot, top_up, top_down))
